# Step 2: Prepare the data
**CS201 Project: Time-aware TSP around elephant crossing zones**

What our group has done in this notebook:
1. Load `MY.txt` (GeoNames, tab-separated, no header).
2. Keeps populated places (feature class **P**) in **Perak (07)** and **Kelantan (03)**.
3. Drops feature codes that are not real, current towns.
4. Selects about 100–200 towns: every town with a population figure, then tops up each district to a minimum count with spread-out towns.
5. Estimates road distances as **haversine distance × 1.3**.

## 1. Ignore -- Folder Sorting

In [13]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')

CODE = "/content/drive/MyDrive/CS201 Project/Code"
MY_TXT = glob.glob(CODE + "/data/**/MY.txt", recursive=True)[0]
OUT_DIR = CODE + "/01_data_prep"
print("Using:", MY_TXT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using: /content/drive/MyDrive/CS201 Project/Code/data/raw/MY.txt


## 2. Settings
| Setting | Value | Why |
| --- | --- | --- |
| `KEEP_CODES` | PPL, PPLA… | Drops PPLX (section of a city), PPLQ (abandoned), PPLH (historical), PPLW (destroyed) |
| `MIN_PER_DISTRICT` | 7 | Every district, including rural elephant areas, is represented |
| `MIN_GAP_KM` | 1.0 | Places closer than 1 km are treated as duplicates |
| `DETOUR_FACTOR` | 1.3 | Road distance ≈ straight-line × 1.3 (no road distances in the dataset) |

In [14]:
STATES = {"07": "Perak", "03": "Kelantan"}
KEEP_CODES = {"PPL", "PPLA", "PPLA2", "PPLA3", "PPLA4", "PPLC", "PPLL", "PPLS", "PPLF", "PPLR"}
MIN_PER_DISTRICT = 7
MIN_GAP_KM = 1.0
DETOUR_FACTOR = 1.3
EARTH_RADIUS_KM = 6371.0

# GeoNames column positions (see Metadata.txt)
COL = {"geonameid": 0, "name": 1, "asciiname": 2, "lat": 4, "lon": 5,
       "fclass": 6, "fcode": 7, "admin1": 10, "admin2": 11, "population": 14}

## 3. Haversine distance
Straight-line distance between two points on the Earth's surface, in km.

In [15]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = p2 - p1
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))

## 4. Load and filter MY.txt
Keeps feature class P in Perak and Kelantan, and reads district (ADM2) names from the same file.

In [16]:
import csv

places, district_names = [], {}
with open(MY_TXT, encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE):
        if len(row) < 19 or row[COL["admin1"]] not in STATES:
            continue
        if row[COL["fcode"]] == "ADM2":
            district_names[row[COL["admin2"]]] = row[COL["asciiname"]].replace("Daerah ", "")
        if row[COL["fclass"]] != "P" or row[COL["fcode"]] not in KEEP_CODES:
            continue
        places.append({
            "geonameid": int(row[COL["geonameid"]]),
            "name": row[COL["asciiname"]] or row[COL["name"]],
            "state": STATES[row[COL["admin1"]]],
            "district_code": row[COL["admin2"]] or "unknown",
            "fcode": row[COL["fcode"]],
            "lat": float(row[COL["lat"]]),
            "lon": float(row[COL["lon"]]),
            "population": int(row[COL["population"]] or 0),
        })

print(f"Candidate populated places in Perak + Kelantan: {len(places)}")
print(f"Districts found: {len(district_names)}")

Candidate populated places in Perak + Kelantan: 3494
Districts found: 20


## 5. Select the towns
- **(a)** every town with a population figure, biggest first
- **(b)** top up each district to `MIN_PER_DISTRICT` using **farthest-point sampling**: repeatedly add the town that is farthest from those already chosen in that district, so coverage is spread out

Towns within `MIN_GAP_KM` of an already-chosen town are skipped as duplicates.

In [17]:
def too_close(town, chosen):
    return any(haversine_km(town["lat"], town["lon"], c["lat"], c["lon"]) < MIN_GAP_KM
               for c in chosen)

chosen = []

# (a) towns with a population figure
for p in sorted(places, key=lambda p: -p["population"]):
    if p["population"] > 0 and not too_close(p, chosen):
        p["reason"] = "has population"
        chosen.append(p)

# (b) district top-up with farthest-point sampling
for d in sorted({p["district_code"] for p in places if p["district_code"] != "unknown"}):
    in_d = [c for c in chosen if c["district_code"] == d]
    pool = [p for p in places if p["district_code"] == d and p not in chosen]
    pool.sort(key=lambda p: (not p["fcode"].startswith("PPLA"), p["geonameid"]))  # admin centres first
    while len(in_d) < MIN_PER_DISTRICT and pool:
        best = pool[0] if not in_d else max(pool, key=lambda p: min(
            haversine_km(p["lat"], p["lon"], c["lat"], c["lon"]) for c in in_d))
        pool.remove(best)
        if too_close(best, chosen):
            continue
        best["reason"] = "district coverage"
        chosen.append(best)
        in_d.append(best)

towns = sorted(chosen, key=lambda p: (p["state"], p["district_code"], p["name"]))
n = len(towns)
print(f"Selected towns: {n}")

Selected towns: 161


## 6. Distance matrix
Road distance estimate = haversine × `DETOUR_FACTOR`. This is a documented assumption.

In [18]:
dist = [[0.0] * n for _ in range(n)]
for i in range(n):
    for j in range(i + 1, n):
        d = haversine_km(towns[i]["lat"], towns[i]["lon"], towns[j]["lat"], towns[j]["lon"]) * DETOUR_FACTOR
        dist[i][j] = dist[j][i] = round(d, 3)
print(f"Distance matrix: {n} x {n}")

Distance matrix: 161 x 161


## 7. Save the output files

In [19]:
os.makedirs(OUT_DIR, exist_ok=True)

with open(os.path.join(OUT_DIR, "towns.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "geonameid", "name", "state", "district_code", "district",
                "feature_code", "latitude", "longitude", "population", "selected_because"])
    for i, t in enumerate(towns):
        w.writerow([i, t["geonameid"], t["name"], t["state"], t["district_code"],
                    district_names.get(t["district_code"], "unknown"), t["fcode"],
                    t["lat"], t["lon"], t["population"], t["reason"]])

with open(os.path.join(OUT_DIR, "distances_km.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id"] + list(range(n)))
    for i in range(n):
        w.writerow([i] + dist[i])

print("Saved towns.csv and distances_km.csv to", OUT_DIR)

Saved towns.csv and distances_km.csv to /content/drive/MyDrive/CS201 Project/Code/01_data_prep


## 8. Summary and sanity check
Compare the sample distances with Google Maps to check the 1.3 detour factor.

In [20]:
by_reason = {}
for t in towns:
    by_reason[t["reason"]] = by_reason.get(t["reason"], 0) + 1

lines = [
    f"Candidate populated places (class P, kept codes) in Perak + Kelantan: {len(places)}",
    f"Selected towns: {n}  {by_reason}",
    f"  Perak: {sum(t['state'] == 'Perak' for t in towns)}, Kelantan: {sum(t['state'] == 'Kelantan' for t in towns)}",
    f"Detour factor: {DETOUR_FACTOR}, min gap: {MIN_GAP_KM} km, min per district: {MIN_PER_DISTRICT}",
    "", "Towns per district:",
]
counts = {}
for t in towns:
    key = (t["state"], district_names.get(t["district_code"], "unknown"))
    counts[key] = counts.get(key, 0) + 1
for (s, d), c in sorted(counts.items()):
    lines.append(f"  {s:9s} {d:28s} {c}")

lines += ["", "Sample road-distance estimates (compare with Google Maps):"]
idx = {t["name"]: i for i, t in enumerate(towns)}
for a, b in [("Ipoh", "Kota Bharu"), ("Gerik", "Kampung Jeli"), ("Ipoh", "Gerik"), ("Kota Bharu", "Kampung Jeli")]:
    if a in idx and b in idx:
        d = dist[idx[a]][idx[b]]
        lines.append(f"  {a} -> {b}: {d:.1f} km (straight line {d / DETOUR_FACTOR:.1f} km)")

text = "\n".join(lines)
with open(os.path.join(OUT_DIR, "summary.txt"), "w") as f:
    f.write(text + "\n")
print(text)

Candidate populated places (class P, kept codes) in Perak + Kelantan: 3494
Selected towns: 161  {'has population': 101, 'district coverage': 60}
  Perak: 88, Kelantan: 73
Detour factor: 1.3, min gap: 1.0 km, min per district: 7

Towns per district:
  Kelantan  Bachok                       7
  Kelantan  Gua Musang                   7
  Kelantan  Jeli                         7
  Kelantan  Kota Bharu                   8
  Kelantan  Kuala Krai                   7
  Kelantan  Machang                      7
  Kelantan  Pasir Mas                    7
  Kelantan  Pasir Puteh                  7
  Kelantan  Tanah Merah                  7
  Kelantan  Tumpat                       7
  Kelantan  unknown                      2
  Perak     Batang Padang                9
  Perak     Hilir Perak                  7
  Perak     Kampar                       7
  Perak     Kerian                       9
  Perak     Kinta                        16
  Perak     Kuala Kangsar                7
  Perak     Larut M